# Particion Temporal y Normalizacion con metodo por Transecto y metodo General

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Partición temporal (train/val/test) y normalización MinMax (solo para DL).
Solo guarda entidades donde todos los splits tengan al menos una muestra.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")
WINDOWS_DIR = os.path.join(BASE_DIR, "windows")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")

INPUT_ML_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "ml")
INPUT_DL_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "dl")
INPUT_ML_GLOBAL = os.path.join(WINDOWS_DIR, "global", "ml")
INPUT_DL_GLOBAL = os.path.join(WINDOWS_DIR, "global", "dl")

OUTPUT_ML_TRANSECT = os.path.join(WINDOWS_PARTITIONED_DIR, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(WINDOWS_PARTITIONED_DIR, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(WINDOWS_PARTITIONED_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(WINDOWS_PARTITIONED_DIR, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = "O3"

# Fechas de corte (sin márgenes adicionales)
TRAIN_START = pd.to_datetime("2024-01-01 00:00:00")
TRAIN_END   = pd.to_datetime("2024-12-31 23:00:00")
VAL_START   = pd.to_datetime("2025-01-01 00:00:00")
VAL_END     = pd.to_datetime("2025-02-01 23:00:00")
TEST_START  = pd.to_datetime("2025-02-04 00:00:00")
INPUT_MARGIN = pd.Timedelta(hours=0)
OUTPUT_MARGIN = pd.Timedelta(hours=0)


def load_entity_data(entity_dir, entity_name):
    X_path = os.path.join(entity_dir, f"{entity_name}_X.npy")
    y_path = os.path.join(entity_dir, f"{entity_name}_y.npy")
    ts_path = os.path.join(entity_dir, f"{entity_name}_timestamps.npy")
    if not (os.path.exists(X_path) and os.path.exists(y_path) and os.path.exists(ts_path)):
        return None, None, None
    X = np.load(X_path)
    y = np.load(y_path)
    timestamps = pd.to_datetime(np.load(ts_path))
    return X, y, timestamps


def split_by_timestamps(X, y, timestamps):
    train_mask = (timestamps >= (TRAIN_START + INPUT_MARGIN)) & (timestamps <= (TRAIN_END - OUTPUT_MARGIN))
    val_mask   = (timestamps >= (VAL_START + INPUT_MARGIN)) & (timestamps <= (VAL_END - OUTPUT_MARGIN))
    test_mask  = timestamps >= (TEST_START + INPUT_MARGIN)
    # Resolver solapamientos
    val_mask = val_mask & ~train_mask
    test_mask = test_mask & ~train_mask & ~val_mask
    return (X[train_mask], y[train_mask]), (X[val_mask], y[val_mask]), (X[test_mask], y[test_mask])


def _scale_X_split(scaler_X, X_split, win_in, n_feat, fit=False):
    if X_split.shape[0] == 0:
        return np.empty((0, win_in, n_feat), dtype=np.float32)
    X_flat = X_split.reshape(-1, n_feat)
    if fit:
        X_scaled = scaler_X.fit_transform(X_flat)
    else:
        X_scaled = scaler_X.transform(X_flat)
    return X_scaled.reshape(X_split.shape[0], win_in, n_feat).astype(np.float32)


def _scale_y_split(scaler_y, y_split, fit=False):
    if y_split.shape[0] == 0:
        return np.empty((0, WINDOW_OUT), dtype=np.float32)
    y_flat = y_split.reshape(-1, 1)
    if fit:
        y_scaled = scaler_y.fit_transform(y_flat)
    else:
        y_scaled = scaler_y.transform(y_flat)
    return y_scaled.reshape(y_split.shape[0], WINDOW_OUT).astype(np.float32)


def normalize_data(X_train, X_val, X_test, y_train, y_val, y_test):
    n_train, win_in, n_feat = X_train.shape
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_train_sc = _scale_X_split(scaler_X, X_train, win_in, n_feat, fit=True)
    X_val_sc   = _scale_X_split(scaler_X, X_val,   win_in, n_feat, fit=False)
    X_test_sc  = _scale_X_split(scaler_X, X_test,  win_in, n_feat, fit=False)
    y_train_sc = _scale_y_split(scaler_y, y_train, fit=True)
    y_val_sc   = _scale_y_split(scaler_y, y_val,   fit=False)
    y_test_sc  = _scale_y_split(scaler_y, y_test,  fit=False)
    return X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y


def save_split_ml(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)
    print(f"    ML guardado: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")


def save_split_dl(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test, scaler_X, scaler_y):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)
    with open(os.path.join(save_dir, "scaler_X.pkl"), "wb") as f:
        pickle.dump(scaler_X, f)
    with open(os.path.join(save_dir, "scaler_y.pkl"), "wb") as f:
        pickle.dump(scaler_y, f)
    print(f"    DL guardado: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")


def process_entity(entity_name, input_ml_dir, input_dl_dir, output_ml_dir, output_dl_dir):
    print(f"\n  Procesando {entity_name}...")
    X_ml, y_ml, ts_ml = load_entity_data(input_ml_dir, entity_name)
    if X_ml is None:
        return
    X_dl, y_dl, ts_dl = load_entity_data(input_dl_dir, entity_name)
    if X_dl is None:
        return
    
    # Verificar que los timestamps coincidan
    if not np.array_equal(ts_ml, ts_dl):
        print(f"    Error: timestamps no coinciden. Se omite.")
        return

    # Calcular máscaras una sola vez
    train_mask = (ts_ml >= (TRAIN_START + INPUT_MARGIN)) & (ts_ml <= (TRAIN_END - OUTPUT_MARGIN))
    val_mask   = (ts_ml >= (VAL_START + INPUT_MARGIN)) & (ts_ml <= (VAL_END - OUTPUT_MARGIN))
    test_mask  = ts_ml >= (TEST_START + INPUT_MARGIN)
    val_mask = val_mask & ~train_mask
    test_mask = test_mask & ~train_mask & ~val_mask

    print(f"    Train mask sum: {train_mask.sum()}")
    print(f"    Val mask sum:   {val_mask.sum()}")
    print(f"    Test mask sum:  {test_mask.sum()}")

    # Aplicar máscaras a ML
    X_train_ml = X_ml[train_mask]
    y_train_ml = y_ml[train_mask]
    X_val_ml   = X_ml[val_mask]
    y_val_ml   = y_ml[val_mask]
    X_test_ml  = X_ml[test_mask]
    y_test_ml  = y_ml[test_mask]

    # Aplicar las MISMAS máscaras a DL
    X_train_dl = X_dl[train_mask]
    y_train_dl = y_dl[train_mask]
    X_val_dl   = X_dl[val_mask]
    y_val_dl   = y_dl[val_mask]
    X_test_dl  = X_dl[test_mask]
    y_test_dl  = y_dl[test_mask]

    # Verificar que todos los splits tengan al menos una muestra
    if len(X_train_ml) == 0 or len(X_val_ml) == 0 or len(X_test_ml) == 0:
        print(f"    ⚠️ Saltando {entity_name}: algún split vacío (train={len(X_train_ml)}, val={len(X_val_ml)}, test={len(X_test_ml)})")
        return

    # Guardar ML
    save_split_ml(output_ml_dir, entity_name, X_train_ml, y_train_ml, X_val_ml, y_val_ml, X_test_ml, y_test_ml)
    
    # Normalizar y guardar DL con el orden correcto
    if len(X_train_dl) > 0:
        X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y = normalize_data(
            X_train_dl, X_val_dl, X_test_dl, y_train_dl, y_val_dl, y_test_dl)
        save_split_dl(output_dl_dir, entity_name, 
                      X_train_sc, y_train_sc, 
                      X_val_sc, y_val_sc, 
                      X_test_sc, y_test_sc, 
                      scaler_X, scaler_y)
    else:
        print(f"    No hay datos DL para {entity_name}")

def process_by_transect():
    print("\n--- Procesando por transecto ---")
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_TRANSECT).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_TRANSECT).glob("*_X.npy")]
    for entity in sorted(set(ml_files) & set(dl_files)):
        process_entity(entity, INPUT_ML_TRANSECT, INPUT_DL_TRANSECT, OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT)


def process_global():
    print("\n--- Procesando global ---")
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_GLOBAL).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_GLOBAL).glob("*_X.npy")]
    for entity in sorted(set(ml_files) & set(dl_files)):
        process_entity(entity, INPUT_ML_GLOBAL, INPUT_DL_GLOBAL, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL)


if __name__ == "__main__":
    print("Partición temporal + normalización")
    process_by_transect()
    process_global()
    print("Proceso completado.")

Partición temporal + normalización

--- Procesando por transecto ---

  Procesando Transecto_1...
    Train mask sum: 8698
    Val mask sum:   749
    Test mask sum:  7807
    ML guardado: train=8698, val=749, test=7807
    DL guardado: train=8698, val=749, test=7807

  Procesando Transecto_2...
    Train mask sum: 8399
    Val mask sum:   687
    Test mask sum:  7665
    ML guardado: train=8399, val=687, test=7665
    DL guardado: train=8399, val=687, test=7665

--- Procesando global ---

  Procesando T1_E1_Alicante...
    Train mask sum: 8604
    Val mask sum:   749
    Test mask sum:  7718
    ML guardado: train=8604, val=749, test=7718
    DL guardado: train=8604, val=749, test=7718

  Procesando T1_E2_Elda...
    Train mask sum: 7762
    Val mask sum:   0
    Test mask sum:  3238
    ⚠️ Saltando T1_E2_Elda: algún split vacío (train=7762, val=0, test=3238)

  Procesando T2_E1_Elche...
    Train mask sum: 8104
    Val mask sum:   687
    Test mask sum:  7525
    ML guardado: train=8